# Modelo Transformer

Notebook de ejemplo para entrenar y probar un modelo Transformer en Keras.


In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


### Modelo


In [ ]:
embed_dim = 32 # Embedding size for each token
num_heads = 2 # Number of attention heads
ff_dim = 32 # Hidden layer size in feed forward network inside transformer

inputs = layers.Input(shape=(maxlen,))
embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)

# Salida en CADA posicion temporal -> (batch, maxlen, vocab_size)
# No hay pooling: predecimos el siguiente token para cada token de la secuencia.

outputs = layers.Dense(vocab_size, activation="softmax")(x)
#

model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

### Entrenamiento


In [ ]:
model.compile(
 optimizer=keras.optimizers.Adam(learning_rate=0.0001),
 loss="sparse_categorical_crossentropy", # sparse_categorical_crossentropy se usa
 # cuando las etiquetas son enteros directamente que representan la distribución
 # one-hot correspondiente.
 metrics=["accuracy"]
)

history = model.fit(
 x_train, y_train, batch_size=32, epochs=15, validation_data=(x_val, y_val)
)

### Pruebas de generación de textos


In [ ]:
def generate_text(model, seed_text, num_words=50, temperature=1.0):
 """
 Genera texto de forma autoregresiva a partir de un texto semilla.

 seed_text : texto inicial en castellano
 num_words : numero de palabras a generar
 temperature : >1 mas creativo/aleatorio; <1 mas conservador/repetitivo
 """
 words = seed_text.lower().split()

 for _ in range(num_words):
 # Tomamos las ultimas maxlen palabras y las convertimos a indices
 context = [word2idx.get(w, 1) for w in words[-maxlen:]]

 # Padding a la izquierda si la secuencia es mas corta que maxlen
 context = [0] * (maxlen - len(context)) + context
 context = np.array(context)[np.newaxis, :] # (1, maxlen)

 # Prediccion: tomamos la distribucion del ULTIMO token -> (vocab_size,)
 probs = model.predict(context, verbose=0)[0, -1]

 # Muestreo con temperatura:
 # dividir log-probs por temperature antes de softmax controla la aleatoriedad
 probs = np.log(probs + 1e-10) / temperature
 probs = np.exp(probs - np.max(probs)) # estabilidad numerica
 probs = probs / probs.sum()

 next_idx = np.random.choice(len(probs), p=probs)
 words.append(idx2word.get(next_idx, ''))

 return ' '.join(words)
